# Qwen3-VL Audio-as-Video Inference Test

This notebook tries to pass an audio file to Qwen3-VL as if it were a `video` input.

It runs two attempts:
1. Directly provide the audio path in a `video` content block.
2. If that fails, convert audio to a black-screen MP4 with the audio track and query that file.

In [ ]:
# If needed, uncomment:
# %pip install -q "transformers>=4.57.0" accelerate torch torchvision torchaudio

In [ ]:
import subprocess
from pathlib import Path

import torch
from transformers import AutoModelForImageTextToText, AutoProcessor

# Pick a smaller checkpoint for local testing.
MODEL_ID = "Qwen/Qwen2.5-VL-3B-Instruct"
# You can switch to a Qwen3-VL checkpoint if available in your environment:
# MODEL_ID = "Qwen/Qwen3-VL-4B-Instruct"

AUDIO_PATH = Path("/path/to/your/audio.wav")
QUESTION = "Describe the audio content. What events or sounds are most likely present?"
MAX_NEW_TOKENS = 192

if not AUDIO_PATH.exists():
    raise FileNotFoundError(f"Set AUDIO_PATH to a real file. Missing: {AUDIO_PATH}")

dtype = torch.bfloat16 if torch.cuda.is_available() else torch.float32
model = AutoModelForImageTextToText.from_pretrained(
    MODEL_ID,
    torch_dtype=dtype,
    device_map="auto",
)
processor = AutoProcessor.from_pretrained(MODEL_ID)

print("Loaded model:", MODEL_ID)
print("Audio path:", AUDIO_PATH)

In [ ]:
def run_qwen_vl(messages, max_new_tokens=192):
    inputs = processor.apply_chat_template(
        messages,
        tokenize=True,
        add_generation_prompt=True,
        return_dict=True,
        return_tensors="pt",
    )
    inputs = {k: v.to(model.device) for k, v in inputs.items()}

    with torch.no_grad():
        generated_ids = model.generate(**inputs, max_new_tokens=max_new_tokens)

    generated_ids_trimmed = [
        out_ids[len(in_ids):] for in_ids, out_ids in zip(inputs["input_ids"], generated_ids)
    ]
    output_text = processor.batch_decode(
        generated_ids_trimmed,
        skip_special_tokens=True,
        clean_up_tokenization_spaces=False,
    )
    return output_text[0]

# Attempt 1: pass raw audio file in a video field.
messages_audio_as_video = [
    {
        "role": "user",
        "content": [
            {"type": "video", "video": str(AUDIO_PATH)},
            {"type": "text", "text": QUESTION},
        ],
    }
]

try:
    out1 = run_qwen_vl(messages_audio_as_video, max_new_tokens=MAX_NEW_TOKENS)
    print("[Attempt 1: audio path as video]\n")
    print(out1)
except Exception as e:
    print("[Attempt 1 failed]")
    print(type(e).__name__, str(e)[:400])

In [ ]:
def audio_to_black_video(audio_path: Path, out_path: Path, width=640, height=360, fps=1):
    cmd = [
        "ffmpeg",
        "-y",
        "-f",
        "lavfi",
        "-i",
        f"color=c=black:s={width}x{height}:r={fps}",
        "-i",
        str(audio_path),
        "-shortest",
        "-c:v",
        "libx264",
        "-pix_fmt",
        "yuv420p",
        "-c:a",
        "aac",
        str(out_path),
    ]
    subprocess.run(cmd, check=True, capture_output=True)

wrapped_video_path = AUDIO_PATH.with_suffix(".audio_wrap.mp4")

try:
    audio_to_black_video(AUDIO_PATH, wrapped_video_path)
    print("Wrapped video created:", wrapped_video_path)

    messages_wrapped = [
        {
            "role": "user",
            "content": [
                {"type": "video", "video": str(wrapped_video_path)},
                {"type": "text", "text": QUESTION},
            ],
        }
    ]

    out2 = run_qwen_vl(messages_wrapped, max_new_tokens=MAX_NEW_TOKENS)
    print("\n[Attempt 2: black-video wrapper + audio track]\n")
    print(out2)
except FileNotFoundError:
    print("ffmpeg is not installed. Install ffmpeg and rerun this cell.")
except subprocess.CalledProcessError as e:
    print("ffmpeg failed:")
    print(e.stderr.decode("utf-8", errors="ignore")[:800])
except Exception as e:
    print("Attempt 2 failed:")
    print(type(e).__name__, str(e)[:400])

## Notes
- Many VL models process visual frames only; if audio is ignored, responses may not reflect actual sound.
- If both attempts fail, use an audio model (e.g., Qwen2-Audio or CLAP/Whisper-style pipelines) for true audio understanding.